# CAC 40 — IVol Exit / Derivative Re-entry Explorer

**Two sliders, one chart:**
- **Exit threshold**: go flat when IVol ≥ X
- **Re-entry days**: re-enter after N consecutive days of falling IVol

Price is **green** when in the market, **red** when flat.

In [8]:
import pandas as pd, numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
import warnings; warnings.filterwarnings('ignore')

PROJECT_ROOT = Path(r"c:\Personal\Business & Investments\Trading portfolio\Cogilator\btest")

In [9]:
# ── Load & align data ──
idx_raw = pd.read_parquet(PROJECT_ROOT / 'equities' / 'indicies.parquet')
cac_px = idx_raw[idx_raw['ticker'] == 'CAC'].set_index('date')[['close', 'volume']].sort_index()

cac_ivol = pd.read_excel(PROJECT_ROOT / 'data' / 'indices_ivol.xlsx', sheet_name='CAC')
cac_ivol.columns = ['date', 'ivol']
cac_ivol['date'] = pd.to_datetime(cac_ivol['date'], errors='coerce')
cac_ivol = cac_ivol.dropna(subset=['date']).set_index('date').sort_index()

common_idx = cac_px.index.intersection(cac_ivol.index)
cac = pd.DataFrame({
    'close': cac_px.loc[common_idx, 'close'],
    'ivol': cac_ivol.loc[common_idx, 'ivol'],
}).sort_index()
cac['ret'] = cac['close'].pct_change()

clean = cac.dropna(subset=['ret']).copy()

# EWMA on IVol: short (5d) and long (20d)
clean['ivol_ema5'] = clean['ivol'].ewm(span=5).mean()
clean['ivol_ema20'] = clean['ivol'].ewm(span=20).mean()

bh_eq = (1 + clean['ret']).cumprod()
bh_total = (bh_eq.iloc[-1] - 1) * 100
bh_sharpe = clean['ret'].mean() / clean['ret'].std() * np.sqrt(252)
bh_mdd = ((bh_eq - bh_eq.cummax()) / bh_eq.cummax()).min() * 100

print(f"Data: {clean.index[0].date()} → {clean.index[-1].date()}  ({len(clean)} days)")
print(f"B&H: {bh_total:+.1f}%  Sharpe: {bh_sharpe:.3f}  MaxDD: {bh_mdd:.1f}%")

Data: 2007-01-03 → 2025-12-12  (4850 days)
B&H: +43.6%  Sharpe: 0.195  MaxDD: -59.2%


In [10]:
# ── Backtest engine: numba-accelerated spike-relative exit ──
import numba as nb
from itertools import product

@nb.njit(cache=True)
def _bt_core(ivol, ret, baseline, ema_f, ema_s,
             spike_pct, exit_confirm, cooldown, confirm):
    """Numba-jit backtest core. Computes position + stats in one pass.
    Returns (sharpe, total%, maxdd%, pct_in%, n_transitions)."""
    spike_mult = 1.0 + spike_pct / 100.0
    n = len(ivol)
    pos = 1
    prev_pos = 1
    days_since_entry = cooldown
    days_ema_below = 0
    days_above_spike = 0
    sum_ret = 0.0
    sum_ret_sq = 0.0
    equity = 1.0
    peak = 1.0
    max_dd = 0.0
    days_in = 0
    n_trans = 0

    for i in range(n):
        iv = ivol[i]
        thr_i = baseline[i] * spike_mult

        if iv >= thr_i:
            days_above_spike += 1
        else:
            days_above_spike = 0

        if ema_f[i] < ema_s[i]:
            days_ema_below += 1
        else:
            days_ema_below = 0

        old_pos = pos
        if pos == 1 and days_above_spike >= exit_confirm and days_since_entry >= cooldown:
            pos = 0
        elif pos == 0 and days_ema_below >= confirm:
            pos = 1
            days_since_entry = 0
        if pos == 1:
            days_since_entry += 1
        if pos != old_pos:
            n_trans += 1

        strat_r = prev_pos * ret[i]
        sum_ret += strat_r
        sum_ret_sq += strat_r * strat_r
        equity *= (1.0 + strat_r)
        if equity > peak:
            peak = equity
        dd = (equity - peak) / peak
        if dd < max_dd:
            max_dd = dd
        if prev_pos > 0:
            days_in += 1
        prev_pos = pos

    total = (equity - 1.0) * 100.0
    mean_r = sum_ret / n
    var_r = sum_ret_sq / n - mean_r * mean_r
    std_r = var_r ** 0.5 if var_r > 0.0 else 1e-10
    sharpe = mean_r / std_r * (252.0 ** 0.5)
    pct_in = days_in / n * 100.0
    return sharpe, total, max_dd * 100.0, pct_in, float(n_trans)


@nb.njit(cache=True)
def _bt_full(ivol, ret, baseline, ema_f, ema_s,
             spike_pct, exit_confirm, cooldown, confirm):
    """Numba-jit backtest returning full arrays for charting."""
    spike_mult = 1.0 + spike_pct / 100.0
    n = len(ivol)
    positions = np.empty(n, dtype=np.float64)
    exit_sig  = np.empty(n, dtype=np.float64)
    entry_sig = np.empty(n, dtype=np.float64)
    exit_level = np.empty(n, dtype=np.float64)

    pos = 1
    days_since_entry = cooldown
    days_ema_below = 0
    days_above_spike = 0

    for i in range(n):
        iv = ivol[i]
        thr_i = baseline[i] * spike_mult

        if iv >= thr_i:
            days_above_spike += 1
        else:
            days_above_spike = 0

        if ema_f[i] < ema_s[i]:
            days_ema_below += 1
        else:
            days_ema_below = 0

        exit_sig[i]  = 0.0 if days_above_spike >= exit_confirm else 1.0
        entry_sig[i] = 1.0 if days_ema_below >= confirm else 0.0
        exit_level[i] = thr_i

        if pos == 1 and days_above_spike >= exit_confirm and days_since_entry >= cooldown:
            pos = 0
        elif pos == 0 and days_ema_below >= confirm:
            pos = 1
            days_since_entry = 0
        if pos == 1:
            days_since_entry += 1
        positions[i] = float(pos)

    return positions, exit_sig, entry_sig, exit_level


# ── Pre-compute all EMA arrays we'll ever need ──
ivol_arr = clean['ivol'].values.astype(np.float64)
ret_arr  = clean['ret'].values.astype(np.float64)

all_spans = sorted(set(range(2, 16)) | set(range(15, 45, 5)) | set(range(20, 125, 5)))
ema_cache = {}
for span in all_spans:
    ema_cache[span] = clean['ivol'].ewm(span=span).mean().values.astype(np.float64)
print(f"Pre-computed {len(ema_cache)} EMA spans: {min(all_spans)}-{max(all_spans)}")

# Warm up numba (first call compiles both kernels)
_ = _bt_core(ivol_arr, ret_arr, ema_cache[60], ema_cache[5], ema_cache[20],
             30.0, 3, 15, 5)
_ = _bt_full(ivol_arr, ret_arr, ema_cache[60], ema_cache[5], ema_cache[20],
             30.0, 3, 15, 5)
print("Numba JIT compiled — each backtest ~10μs")


def bt_ivol_spike(df, exit_ema=60, spike_pct=30, exit_confirm=3,
                  ema_fast=5, ema_slow=20, cooldown=15, confirm=5):
    """Full backtest returning DataFrame + stats (numba-accelerated)."""
    res = df[['close', 'ret', 'ivol']].copy()
    baseline = res['ivol'].ewm(span=exit_ema).mean().values.astype(np.float64)
    ema_f_arr = res['ivol'].ewm(span=ema_fast).mean().values.astype(np.float64)
    ema_s_arr = res['ivol'].ewm(span=ema_slow).mean().values.astype(np.float64)
    ivol = res['ivol'].values.astype(np.float64)
    ret_a = res['ret'].values.astype(np.float64)

    positions, exit_sig, entry_sig, exit_lev = _bt_full(
        ivol, ret_a, baseline, ema_f_arr, ema_s_arr,
        float(spike_pct), exit_confirm, cooldown, confirm)

    res['position']  = positions
    res['exit_sig']  = exit_sig
    res['entry_sig'] = entry_sig
    res['exit_level'] = exit_lev
    res['strat_ret'] = res['position'].shift(1).fillna(1) * res['ret']
    res['equity_strat'] = (1 + res['strat_ret']).cumprod()
    sr = res['strat_ret']
    eq = res['equity_strat']
    sharpe = sr.mean() / sr.std() * np.sqrt(252) if sr.std() > 0 else 0
    mdd = ((eq - eq.cummax()) / eq.cummax()).min()
    total = (eq.iloc[-1] - 1) * 100
    pct_in = (res['position'].shift(1).fillna(1) > 0).sum() / len(res) * 100
    stats = dict(sharpe=round(sharpe, 3), total=round(total, 1),
                 maxdd=round(mdd * 100, 1), pct_in=round(pct_in, 1))
    return res, stats

Pre-computed 35 EMA spans: 2-120
Numba JIT compiled — each backtest ~10μs


In [11]:
# ── Quick validation: numba vs pandas backtest match ──
r_fast = _bt_core(ivol_arr, ret_arr, ema_cache[60], ema_cache[5], ema_cache[20],
                  30.0, 3, 15, 5)
_, s_slow = bt_ivol_spike(clean, exit_ema=60, spike_pct=30, exit_confirm=3,
                           ema_fast=5, ema_slow=20, cooldown=15, confirm=5)
print(f"Numba:  Sharpe={r_fast[0]:.3f}  Total={r_fast[1]:+.1f}%  MaxDD={r_fast[2]:.1f}%  In={r_fast[3]:.1f}%")
print(f"Pandas: Sharpe={s_slow['sharpe']:.3f}  Total={s_slow['total']:+.1f}%  MaxDD={s_slow['maxdd']:.1f}%  In={s_slow['pct_in']:.1f}%")
print(f"Match: {'YES' if abs(r_fast[0] - s_slow['sharpe']) < 0.01 else 'NO'}")

# Time comparison
import time
t0 = time.perf_counter()
for _ in range(1000):
    _bt_core(ivol_arr, ret_arr, ema_cache[60], ema_cache[5], ema_cache[20], 30.0, 3, 15, 5)
t_numba = (time.perf_counter() - t0) / 1000 * 1e6

t0 = time.perf_counter()
for _ in range(100):
    bt_ivol_spike(clean, exit_ema=60, spike_pct=30, exit_confirm=3, ema_fast=5, confirm=5)
t_pandas = (time.perf_counter() - t0) / 100 * 1e6

print(f"\nNumba: {t_numba:.0f} μs/call  |  Pandas: {t_pandas:.0f} μs/call  |  Speedup: {t_pandas/t_numba:.0f}x")

Numba:  Sharpe=0.243  Total=+72.3%  MaxDD=-52.8%  In=96.5%
Pandas: Sharpe=0.243  Total=+72.3%  MaxDD=-52.8%  In=96.5%
Match: YES

Numba: 7 μs/call  |  Pandas: 1181 μs/call  |  Speedup: 169x


In [12]:
# ── Fast optimisation: numba + pre-computed EMAs ──
import time

# Full parameter grid — ALL parameters included
p_exit_ema     = [20, 30, 40, 50, 60, 80, 100, 120]       # baseline window
p_spike_pct    = list(range(15, 85, 5))                     # 15,20,...,80
p_exit_confirm = list(range(1, 8))                          # 1..7 days
p_cooldown     = [5, 10, 15, 20, 30]                        # anti-whipsaw
p_ema_fast     = [2, 3, 4, 5, 6, 7, 8, 10]                 # re-entry fast EMA
p_ema_slow     = [15, 20, 25, 30]                           # re-entry slow EMA
p_confirm      = [1, 2, 3, 5, 7, 10, 15]                   # re-entry confirm

combos = list(product(p_exit_ema, p_spike_pct, p_exit_confirm,
                       p_cooldown, p_ema_fast, p_ema_slow, p_confirm))
print(f"Grid: {len(combos):,} combinations")
print(f"  exit_ema:     {p_exit_ema}")
print(f"  spike_pct:    {p_spike_pct[0]}-{p_spike_pct[-1]} step 5")
print(f"  exit_confirm: {p_exit_confirm}")
print(f"  cooldown:     {p_cooldown}")
print(f"  ema_fast:     {p_ema_fast}")
print(f"  ema_slow:     {p_ema_slow}")
print(f"  confirm:      {p_confirm}")

t0 = time.perf_counter()
results = np.empty((len(combos), 12), dtype=np.float64)

for idx, (ex_ema, sp, ex_cf, cd, ef, es, cf) in enumerate(combos):
    sh, tot, mdd, pin, ntr = _bt_core(
        ivol_arr, ret_arr, ema_cache[ex_ema], ema_cache[ef], ema_cache[es],
        float(sp), ex_cf, cd, cf)
    results[idx] = [ex_ema, sp, ex_cf, cd, ef, es, cf, sh, tot, mdd, pin, ntr]

elapsed = time.perf_counter() - t0
print(f"\nCompleted in {elapsed:.1f}s  ({elapsed/len(combos)*1e6:.1f} μs/backtest)")

opt = pd.DataFrame(results, columns=[
    'exit_ema', 'spike_pct', 'exit_confirm', 'cooldown',
    'ema_fast', 'ema_slow', 'confirm',
    'sharpe', 'total', 'maxdd', 'pct_in', 'n_trans'
])
for c in ['exit_ema','spike_pct','exit_confirm','cooldown','ema_fast','ema_slow','confirm','n_trans']:
    opt[c] = opt[c].astype(int)

# ── Filter & rank ──
viable = opt[(opt['pct_in'] > 50) & (opt['sharpe'] > 0)].copy()
viable['score'] = viable['sharpe'] - (viable['maxdd'].abs() / 100) * 0.5

show_cols = ['exit_ema','spike_pct','exit_confirm','cooldown',
             'ema_fast','ema_slow','confirm',
             'sharpe','total','maxdd','pct_in','n_trans']
fmt = {'sharpe':'{:.3f}','total':'{:+.1f}%','maxdd':'{:.1f}%','pct_in':'{:.0f}%'}

print(f"\n{'='*95}")
print(f"Viable (in-market > 50%, Sharpe > 0): {len(viable):,} / {len(opt):,}")
print(f"{'='*95}")

print(f"\nTop 20 by Sharpe:")
display(viable.nlargest(20, 'sharpe')[show_cols].reset_index(drop=True)
    .style.format(fmt)
    .bar(subset=['sharpe'], color='#66ffcc')
    .bar(subset=['total'], color='#8cb4ff')
    .background_gradient(subset=['maxdd'], cmap='RdYlGn'))

print(f"\nTop 10 by Total Return:")
display(viable.nlargest(10, 'total')[show_cols].reset_index(drop=True)
    .style.format(fmt).bar(subset=['total'], color='#8cb4ff'))

print(f"\nTop 10 Balanced (Sharpe − 0.5×|MaxDD|):")
display(viable.nlargest(10, 'score')[show_cols + ['score']].reset_index(drop=True)
    .style.format({**fmt, 'score':'{:.3f}'}).bar(subset=['score'], color='#ffb347'))

Grid: 878,080 combinations
  exit_ema:     [20, 30, 40, 50, 60, 80, 100, 120]
  spike_pct:    15-80 step 5
  exit_confirm: [1, 2, 3, 4, 5, 6, 7]
  cooldown:     [5, 10, 15, 20, 30]
  ema_fast:     [2, 3, 4, 5, 6, 7, 8, 10]
  ema_slow:     [15, 20, 25, 30]
  confirm:      [1, 2, 3, 5, 7, 10, 15]

Completed in 6.1s  (6.9 μs/backtest)

Viable (in-market > 50%, Sharpe > 0): 868,083 / 878,080

Top 20 by Sharpe:


,exit_ema,spike_pct,exit_confirm,cooldown,ema_fast,ema_slow,confirm,sharpe,total,maxdd,pct_in,n_trans
0,80,30,3,5,2,15,1,0.315,+125.0%,-51.0%,98%,20
1,60,30,3,5,5,20,7,0.303,+112.5%,-44.6%,96%,14
2,60,30,3,10,5,20,7,0.303,+112.5%,-44.6%,96%,14
3,60,30,3,15,5,20,7,0.303,+112.5%,-44.6%,96%,14
4,60,30,3,20,5,20,7,0.303,+112.5%,-44.6%,96%,14
5,60,30,3,30,5,20,7,0.303,+112.5%,-44.6%,96%,14
6,80,30,3,5,5,20,7,0.303,+112.5%,-44.6%,96%,14
7,80,30,3,10,5,20,7,0.303,+112.5%,-44.6%,96%,14
8,80,30,3,15,5,20,7,0.303,+112.5%,-44.6%,96%,14
9,80,30,3,20,5,20,7,0.303,+112.5%,-44.6%,96%,14



Top 10 by Total Return:


,exit_ema,spike_pct,exit_confirm,cooldown,ema_fast,ema_slow,confirm,sharpe,total,maxdd,pct_in,n_trans
0,80,30,3,5,2,15,1,0.315,+125.0%,-51.0%,98%,20
1,100,35,3,5,2,15,1,0.302,+114.4%,-51.0%,98%,22
2,120,35,3,5,2,15,1,0.299,+112.5%,-51.0%,98%,24
3,60,30,3,5,5,20,7,0.303,+112.5%,-44.6%,96%,14
4,60,30,3,10,5,20,7,0.303,+112.5%,-44.6%,96%,14
5,60,30,3,15,5,20,7,0.303,+112.5%,-44.6%,96%,14
6,60,30,3,20,5,20,7,0.303,+112.5%,-44.6%,96%,14
7,60,30,3,30,5,20,7,0.303,+112.5%,-44.6%,96%,14
8,80,30,3,5,5,20,7,0.303,+112.5%,-44.6%,96%,14
9,80,30,3,10,5,20,7,0.303,+112.5%,-44.6%,96%,14



Top 10 Balanced (Sharpe − 0.5×|MaxDD|):


,exit_ema,spike_pct,exit_confirm,cooldown,ema_fast,ema_slow,confirm,sharpe,total,maxdd,pct_in,n_trans,score
0,60,30,3,5,5,20,7,0.303,+112.5%,-44.6%,96%,14,0.080
1,60,30,3,10,5,20,7,0.303,+112.5%,-44.6%,96%,14,0.080
2,60,30,3,15,5,20,7,0.303,+112.5%,-44.6%,96%,14,0.080
3,60,30,3,20,5,20,7,0.303,+112.5%,-44.6%,96%,14,0.080
4,60,30,3,30,5,20,7,0.303,+112.5%,-44.6%,96%,14,0.080
5,80,30,3,5,5,20,7,0.303,+112.5%,-44.6%,96%,14,0.080
6,80,30,3,10,5,20,7,0.303,+112.5%,-44.6%,96%,14,0.080
7,80,30,3,15,5,20,7,0.303,+112.5%,-44.6%,96%,14,0.080
8,80,30,3,20,5,20,7,0.303,+112.5%,-44.6%,96%,14,0.080
9,80,30,3,30,5,20,7,0.303,+112.5%,-44.6%,96%,14,0.080


In [13]:
# ── Diagnostic: inspect 2008-2009 period ──
dbg_exit_ema, dbg_spike, dbg_ef, dbg_cf = 60, 30, 5, 5
res_dbg, stats_dbg = bt_ivol_spike(clean, exit_ema=dbg_exit_ema, spike_pct=dbg_spike,
                                    ema_fast=dbg_ef, confirm=dbg_cf)
diag = res_dbg.loc['2008-01':'2009-09', ['close', 'ivol', 'position', 'exit_level']].copy()
diag['ema_f'] = clean['ivol'].ewm(span=dbg_ef).mean().loc[diag.index]
diag['ema20'] = clean.loc[diag.index, 'ivol_ema20']
diag['pos_chg'] = diag['position'].diff().fillna(0).astype(int)

transitions = diag[diag['pos_chg'] != 0].index
flat_days = diag[diag['position'] == 0].index
flat_samples = flat_days[::10]

show_rows = set()
for t in transitions:
    loc = diag.index.get_loc(t)
    show_rows.update(range(max(0, loc-3), min(len(diag), loc+4)))
for t in flat_samples:
    loc = diag.index.get_loc(t)
    show_rows.add(loc)
show_rows = sorted(show_rows)

n_exits = (diag['pos_chg'] == -1).sum()
n_entries = (diag['pos_chg'] == 1).sum()
print(f"Config: exit EMA-{dbg_exit_ema} × {1+dbg_spike/100:.2f}, EMA-fast={dbg_ef}, confirm={dbg_cf}d")
print(f"Period: Jan 2008 - Sep 2009  |  Exits: {n_exits}  Entries: {n_entries}")
print(f"Stats: {stats_dbg}\n")
diag.iloc[show_rows].style.format({
    'close': '{:.0f}', 'ivol': '{:.1f}', 'exit_level': '{:.1f}',
    'ema_f': '{:.1f}', 'ema20': '{:.1f}'
}).apply(lambda row: ['background: #ffe0e0' if row['pos_chg'] == -1
                      else 'background: #e0ffe0' if row['pos_chg'] == 1
                      else '' for _ in row], axis=1)

Config: exit EMA-60 × 1.30, EMA-fast=5, confirm=5d
Period: Jan 2008 - Sep 2009  |  Exits: 1  Entries: 1
Stats: {'sharpe': np.float64(0.243), 'total': np.float64(72.3), 'maxdd': np.float64(-52.8), 'pct_in': np.float64(96.5)}



,close,ivol,position,exit_level,ema_f,ema20,pos_chg
date,,,,,,,
2008-09-26 00:00:00,4163,29.7,1.000000,31.2,28.3,26.2,0
2008-09-29 00:00:00,3953,32.6,1.000000,31.5,29.7,26.8,0
2008-09-30 00:00:00,4032,32.7,1.000000,31.9,30.7,27.4,0
2008-10-01 00:00:00,4055,33.5,0.000000,32.3,31.6,28.0,-1
2008-10-02 00:00:00,3963,34.2,0.000000,32.7,32.5,28.6,0
2008-10-03 00:00:00,4081,32.8,0.000000,33.0,32.6,29.0,0
2008-10-06 00:00:00,3712,39.6,0.000000,33.6,34.9,30.0,0
2008-10-15 00:00:00,3381,48.6,0.000000,38.9,45.0,37.8,0
2008-10-29 00:00:00,3403,49.5,0.000000,46.6,50.8,46.0,0


In [ ]:
# ── Interactive chart: 7 sliders, 4-panel chart  ─────────────────────────────
# ── Dark navy theme ──────────────────────────────────────────────────────────
_BG      = '#111827'   # dark navy (paper)
_PBG     = '#1a2235'   # slightly lighter (plot area)
_GRID    = '#1e293b'   # subtle grid lines

_C_LONG  = '#60a5fa'   # blue-400   – price while Long
_C_FLAT  = '#f87171'   # red-400    – price while Flat
_C_STRAT = '#34d399'   # emerald    – strategy equity
_C_BH    = '#6b7280'   # gray-500   – buy & hold
_C_IVOL  = '#fb923c'   # orange-400 – IVol dots
_C_EMAF  = '#fbbf24'   # amber-400  – EMA fast
_C_EMAS  = '#a78bfa'   # violet-400 – EMA slow
_C_EXIT  = '#38bdf8'   # sky-400    – exit level
_C_ESIG  = '#f87171'   # red fill   – exit signal
_C_NSIG  = '#34d399'   # green fill – entry signal

# Auto-load best params from optimisation (if available)
_best = viable.nlargest(1, 'sharpe').iloc[0] if len(viable) > 0 else None
_bp = {k: int(_best[k]) for k in ['exit_ema','spike_pct','exit_confirm','cooldown',
       'ema_fast','ema_slow','confirm']} if _best is not None else dict(
       exit_ema=60, spike_pct=30, exit_confirm=3, cooldown=15, ema_fast=5, ema_slow=20, confirm=5)
print(f"Slider defaults → best optimisation result: {_bp}")

def _build_fig(exit_ema, spike_pct, exit_cf, cooldown, ef, es, cf):
    res, stats = bt_ivol_spike(clean, exit_ema=exit_ema, spike_pct=spike_pct,
                                exit_confirm=exit_cf, ema_fast=ef, ema_slow=es,
                                cooldown=cooldown, confirm=cf)
    exp = res['position'].shift(1).fillna(1)
    in_mask = exp > 0

    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04,
        row_heights=[0.32, 0.20, 0.26, 0.22])

    # Row 1: Price
    fig.add_trace(go.Scatter(x=clean.index, y=clean['close'],
        line=dict(color='#374151', width=1), showlegend=False, hoverinfo='skip'), row=1, col=1)
    fig.add_trace(go.Scatter(x=res.index, y=res['close'].where(in_mask),
        name='Long', line=dict(color=_C_LONG, width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=res.index, y=res['close'].where(~in_mask),
        name='Flat', line=dict(color=_C_FLAT, width=2)), row=1, col=1)

    # Row 2: Equity
    fig.add_trace(go.Scatter(x=clean.index, y=bh_eq, name='Buy & Hold',
        line=dict(color=_C_BH, width=1.5, dash='dash')), row=2, col=1)
    fig.add_trace(go.Scatter(x=res.index, y=res['equity_strat'], name='Strategy',
        line=dict(color=_C_STRAT, width=2)), row=2, col=1)

    # Row 3: IVol
    fig.add_trace(go.Scatter(x=clean.index, y=clean['ivol'], name='IVol',
        line=dict(color=_C_IVOL, width=1, dash='dot')), row=3, col=1)
    ema_f = clean['ivol'].ewm(span=ef).mean()
    ema_s = clean['ivol'].ewm(span=es).mean()
    fig.add_trace(go.Scatter(x=clean.index, y=ema_f, name=f'EMA-{ef}',
        line=dict(color=_C_EMAF, width=1.5)), row=3, col=1)
    fig.add_trace(go.Scatter(x=clean.index, y=ema_s, name=f'EMA-{es}',
        line=dict(color=_C_EMAS, width=1.5)), row=3, col=1)
    fig.add_trace(go.Scatter(x=res.index, y=res['exit_level'],
        name=f'Exit (EMA-{exit_ema}×{1+spike_pct/100:.2f})',
        line=dict(color=_C_EXIT, width=1.5, dash='dash')), row=3, col=1)

    # Row 4: Signals
    fig.add_trace(go.Scatter(x=res.index, y=res['exit_sig'],
        name=f'Vol≫spike ({exit_cf}d)', line=dict(color=_C_ESIG, width=1.2),
        fill='tozeroy', fillcolor='rgba(248,113,113,0.12)'), row=4, col=1)
    fig.add_trace(go.Scatter(x=res.index, y=res['entry_sig'] * 0.9,
        name=f'Entry (EMA {ef}<{es} {cf}d)', line=dict(color=_C_NSIG, width=1.2),
        fill='tozeroy', fillcolor='rgba(52,211,153,0.12)'), row=4, col=1)

    n_tr = (res['position'].diff().fillna(0) != 0).sum()
    title_text = (
        f"<b>Exit: IVol > EMA-{exit_ema}×{1+spike_pct/100:.0%} for {exit_cf}d (cd={cooldown})  |  "
        f"Re-enter: EMA-{ef} < EMA-{es} for {cf}d</b><br>"
        f"<span style='font-size:13px; color:#94a3b8'>"
        f"Sharpe <b style='color:#34d399'>{stats['sharpe']:.3f}</b>  ·  "
        f"Total <b style='color:#60a5fa'>{stats['total']:+.1f}%</b>  ·  "
        f"MaxDD <b style='color:#f87171'>{stats['maxdd']:.1f}%</b>  ·  "
        f"In {stats['pct_in']:.0f}%  ·  "
        f"Trades {n_tr:.0f}"
        f"&nbsp;&nbsp;│&nbsp;&nbsp;"
        f"B&H: {bh_sharpe:.3f} / {bh_total:+.1f}% / {bh_mdd:.1f}%</span>"
    )

    fig.update_layout(
        height=1050, hovermode='x unified',
        title=dict(text=title_text, font=dict(size=14, color='#e2e8f0')),
        legend=dict(orientation='h', y=-0.05, font=dict(color='#94a3b8')),
        margin=dict(t=100, b=60),
        paper_bgcolor=_BG, plot_bgcolor=_PBG,
        font=dict(color='#94a3b8'))
    for r in range(1, 5):
        fig.update_xaxes(gridcolor=_GRID, zeroline=False, row=r, col=1)
        fig.update_yaxes(gridcolor=_GRID, zeroline=False, row=r, col=1)
    fig.update_yaxes(title_text='Price', row=1, col=1)
    fig.update_yaxes(title_text='Equity', type='log', row=2, col=1)
    fig.update_yaxes(title_text='IVol Level', row=3, col=1)
    fig.update_yaxes(title_text='Signals', range=[-0.1, 1.15], dtick=1, row=4, col=1)
    return fig

# ── Sliders ──
w = {'description_width': '140px'}
ly = widgets.Layout(width='90%')

sl_exit_ema = widgets.IntSlider(value=_bp['exit_ema'], min=20, max=120, step=5,
    description='Exit baseline EMA', continuous_update=False, style=w, layout=ly)
sl_spike = widgets.IntSlider(value=_bp['spike_pct'], min=10, max=100, step=5,
    description='Spike % above EMA', continuous_update=False, style=w, layout=ly)
sl_exit_cf = widgets.IntSlider(value=_bp['exit_confirm'], min=1, max=10, step=1,
    description='Exit confirm days', continuous_update=False, style=w, layout=ly)
sl_cooldown = widgets.IntSlider(value=_bp['cooldown'], min=1, max=60, step=1,
    description='Cooldown days', continuous_update=False, style=w, layout=ly)

sl_ema_fast = widgets.IntSlider(value=_bp['ema_fast'], min=2, max=15, step=1,
    description='Re-entry EMA-fast', continuous_update=False, style=w, layout=ly)
sl_ema_slow = widgets.IntSlider(value=_bp['ema_slow'], min=10, max=40, step=5,
    description='Re-entry EMA-slow', continuous_update=False, style=w, layout=ly)
sl_confirm = widgets.IntSlider(value=_bp['confirm'], min=1, max=20, step=1,
    description='Re-entry confirm', continuous_update=False, style=w, layout=ly)

out = widgets.Output()

def _update(_=None):
    with out:
        from IPython.display import clear_output
        clear_output(wait=True)
        _build_fig(sl_exit_ema.value, sl_spike.value, sl_exit_cf.value,
                   sl_cooldown.value, sl_ema_fast.value, sl_ema_slow.value,
                   sl_confirm.value).show()

for s in [sl_exit_ema, sl_spike, sl_exit_cf, sl_cooldown,
          sl_ema_fast, sl_ema_slow, sl_confirm]:
    s.observe(_update, 'value')

# ── "Load Best" buttons ──
def _load_best(rank=0):
    def _cb(_):
        row = viable.nlargest(rank + 1, 'sharpe').iloc[rank]
        sl_exit_ema.value = int(row['exit_ema'])
        sl_spike.value = int(row['spike_pct'])
        sl_exit_cf.value = int(row['exit_confirm'])
        sl_cooldown.value = int(row['cooldown'])
        sl_ema_fast.value = int(row['ema_fast'])
        sl_ema_slow.value = int(row['ema_slow'])
        sl_confirm.value = int(row['confirm'])
    return _cb

btn_best1 = widgets.Button(description='▶ Best Sharpe', button_style='success',
    layout=widgets.Layout(width='140px'))
btn_best2 = widgets.Button(description='▶ 2nd Best', button_style='info',
    layout=widgets.Layout(width='120px'))
btn_best1.on_click(_load_best(0))
btn_best2.on_click(_load_best(1))

_update()

exit_box = widgets.VBox([
    widgets.HTML('<b style="font-size:13px; color:#f87171">EXIT parameters</b>'),
    sl_exit_ema, sl_spike, sl_exit_cf, sl_cooldown])
entry_box = widgets.VBox([
    widgets.HTML('<b style="font-size:13px; color:#34d399">RE-ENTRY parameters</b>'),
    sl_ema_fast, sl_ema_slow, sl_confirm])
btn_row = widgets.HBox([btn_best1, btn_best2],
    layout=widgets.Layout(gap='8px'))

display(widgets.VBox([btn_row, exit_box, entry_box, out]))

Slider defaults → best optimisation result: {'exit_ema': 80, 'spike_pct': 30, 'exit_confirm': 3, 'cooldown': 5, 'ema_fast': 2, 'ema_slow': 15, 'confirm': 1}


---
# Enhanced Multi-Signal Optimisation

The basic strategy above only uses **close price + IVol**. We have full **OHLCV** data — let's exploit it.

**New signal dimensions:**
| Signal | Source | Rationale |
|--------|--------|-----------|
| Parkinson RVol | High/Low | Intraday range captures realised vol far better than close-to-close |
| IVol − RVol spread | IVol vs Parkinson | Vol risk premium — large spread = market pricing in fear beyond what's realised |
| Volume z-score | Volume | Abnormal volume confirms panic / capitulation |
| Price momentum | Close | Avoid re-entering downtrends (simple MA filter) |
| Intraday range | (H−L)/C | Direct stress/fear gauge |

Each signal adds an **optional filter** to the exit and/or entry logic.

In [15]:
# ── Load full OHLCV + compute derived signals ──
cac_ohlcv = idx_raw[idx_raw['ticker'] == 'CAC'].set_index('date')[
    ['open', 'high', 'low', 'close', 'volume']].sort_index()

# Merge with IVol on common dates
enh = pd.DataFrame({
    'open':   cac_ohlcv.loc[common_idx, 'open'],
    'high':   cac_ohlcv.loc[common_idx, 'high'],
    'low':    cac_ohlcv.loc[common_idx, 'low'],
    'close':  cac_ohlcv.loc[common_idx, 'close'],
    'volume': cac_ohlcv.loc[common_idx, 'volume'],
    'ivol':   cac_ivol.loc[common_idx, 'ivol'],
}).sort_index()
enh['ret'] = enh['close'].pct_change()
enh = enh.dropna(subset=['ret']).copy()

# ── Parkinson realised volatility (annualised, rolling) ──
log_hl = np.log(enh['high'] / enh['low'])
parkinson_factor = 1.0 / (4.0 * np.log(2))
for win in [5, 10, 20]:
    enh[f'rvol_park_{win}'] = np.sqrt(
        parkinson_factor * (log_hl ** 2).rolling(win).mean() * 252
    ) * 100  # in % to match IVol scale

# ── IVol − RVol spread (vol risk premium) ──
for win in [10, 20]:
    enh[f'vol_spread_{win}'] = enh['ivol'] - enh[f'rvol_park_{win}']

# ── Volume z-score (rolling) ──
for win in [20, 50]:
    vol_mean = enh['volume'].rolling(win).mean()
    vol_std  = enh['volume'].rolling(win).std()
    enh[f'vol_z_{win}'] = (enh['volume'] - vol_mean) / vol_std.replace(0, np.nan)

# ── Intraday range as % of close ──
enh['intraday_range'] = (enh['high'] - enh['low']) / enh['close'] * 100

# ── Price momentum: SMA ratio ──
for win in [20, 50, 100]:
    enh[f'sma_{win}'] = enh['close'].rolling(win).mean()
enh['mom_20_50'] = enh['sma_20'] / enh['sma_50'] - 1  # positive = uptrend
enh['mom_20_100'] = enh['sma_20'] / enh['sma_100'] - 1

# ── IVol EMAs (reuse for consistency) ──
enh['ivol_ema5']  = enh['ivol'].ewm(span=5).mean()
enh['ivol_ema20'] = enh['ivol'].ewm(span=20).mean()

# Drop warmup rows
enh_clean = enh.iloc[100:].copy()  # 100 days warmup for SMA-100

print(f"Enhanced data: {enh_clean.index[0].date()} → {enh_clean.index[-1].date()}  ({len(enh_clean)} days)")
print(f"\nDerived signals:")
for col in ['rvol_park_10', 'rvol_park_20', 'vol_spread_10', 'vol_spread_20',
            'vol_z_20', 'vol_z_50', 'intraday_range', 'mom_20_50', 'mom_20_100']:
    print(f"  {col:20s}  mean={enh_clean[col].mean():+8.3f}  std={enh_clean[col].std():8.3f}")

# B&H stats on the enhanced (trimmed) dataset
enh_bh_eq = (1 + enh_clean['ret']).cumprod()
enh_bh_total = (enh_bh_eq.iloc[-1] - 1) * 100
enh_bh_sharpe = enh_clean['ret'].mean() / enh_clean['ret'].std() * np.sqrt(252)
enh_bh_mdd = ((enh_bh_eq - enh_bh_eq.cummax()) / enh_bh_eq.cummax()).min() * 100
print(f"\nB&H (trimmed): {enh_bh_total:+.1f}%  Sharpe: {enh_bh_sharpe:.3f}  MaxDD: {enh_bh_mdd:.1f}%")

Enhanced data: 2007-05-30 → 2025-12-12  (4750 days)

Derived signals:
  rvol_park_10          mean= +14.716  std=   7.688
  rvol_park_20          mean= +14.916  std=   7.296
  vol_spread_10         mean=  +4.999  std=   3.899
  vol_spread_20         mean=  +4.800  std=   3.456
  vol_z_20              mean=  +0.000  std=   1.048
  vol_z_50              mean=  -0.011  std=   1.065
  intraday_range        mean=  +1.447  std=   0.971
  mom_20_50             mean=  +0.001  std=   0.029
  mom_20_100            mean=  +0.003  std=   0.051

B&H (trimmed): +33.2%  Sharpe: 0.179  MaxDD: -59.2%


In [16]:
# ── Enhanced numba backtest: IVol + RVol spread + Volume + Momentum filters ──

@nb.njit(cache=True)
def _bt_enhanced(ivol, ret, baseline, ema_f, ema_s,
                 vol_spread, vol_z, mom_signal, intraday_range,
                 spike_pct, exit_confirm, cooldown, confirm,
                 # New filter params:
                 spread_exit_thr,   # exit also if vol_spread > thr (0=disabled)
                 vol_z_exit_thr,    # exit also if vol_z > thr (0=disabled)
                 mom_entry_thr,     # only re-enter if mom > thr (-999=disabled)
                 range_exit_mult):  # exit if intraday_range > mult × rolling mean (0=disabled)
    """Enhanced backtest with multi-signal exit/entry filters."""
    spike_mult = 1.0 + spike_pct / 100.0
    n = len(ivol)
    pos = 1
    prev_pos = 1
    days_since_entry = cooldown
    days_ema_below = 0
    days_above_spike = 0
    sum_ret = 0.0
    sum_ret_sq = 0.0
    equity = 1.0
    peak = 1.0
    max_dd = 0.0
    days_in = 0
    n_trans = 0

    # Rolling mean of intraday_range (simple 20d)
    range_buf = np.empty(20)
    range_buf[:] = 0.0
    range_idx = 0
    range_sum = 0.0

    for i in range(n):
        iv = ivol[i]
        thr_i = baseline[i] * spike_mult

        # Update rolling range mean
        old_val = range_buf[range_idx]
        range_buf[range_idx] = intraday_range[i]
        range_sum = range_sum - old_val + intraday_range[i]
        range_idx = (range_idx + 1) % 20
        range_mean = range_sum / min(i + 1, 20)

        # IVol spike detection
        if iv >= thr_i:
            days_above_spike += 1
        else:
            days_above_spike = 0

        # EMA crossover for re-entry
        if ema_f[i] < ema_s[i]:
            days_ema_below += 1
        else:
            days_ema_below = 0

        # ── EXIT logic: OR of multiple conditions ──
        exit_triggered = False

        # Original: IVol spike above baseline
        if days_above_spike >= exit_confirm and days_since_entry >= cooldown:
            exit_triggered = True

        # NEW: Vol risk premium spike (IVol >> RVol)
        if spread_exit_thr > 0.0 and vol_spread[i] > spread_exit_thr:
            exit_triggered = True

        # NEW: Volume z-score spike (panic selling)
        if vol_z_exit_thr > 0.0 and vol_z[i] > vol_z_exit_thr:
            # Only exit on high vol_z when IVol is also elevated (above baseline)
            if iv > baseline[i]:
                exit_triggered = True

        # NEW: Extreme intraday range
        if range_exit_mult > 0.0 and i >= 20:
            if intraday_range[i] > range_mean * range_exit_mult:
                if iv > baseline[i]:
                    exit_triggered = True

        # ── ENTRY logic: AND of conditions ──
        entry_ok = True

        # Original: EMA fast < slow for N days
        if days_ema_below < confirm:
            entry_ok = False

        # NEW: Momentum filter — only re-enter if price trend is positive enough
        if mom_entry_thr > -900.0 and mom_signal[i] < mom_entry_thr:
            entry_ok = False

        # ── State transitions ──
        old_pos = pos
        if pos == 1 and exit_triggered and days_since_entry >= cooldown:
            pos = 0
        elif pos == 0 and entry_ok:
            pos = 1
            days_since_entry = 0
        if pos == 1:
            days_since_entry += 1
        if pos != old_pos:
            n_trans += 1

        strat_r = prev_pos * ret[i]
        sum_ret += strat_r
        sum_ret_sq += strat_r * strat_r
        equity *= (1.0 + strat_r)
        if equity > peak:
            peak = equity
        dd = (equity - peak) / peak
        if dd < max_dd:
            max_dd = dd
        if prev_pos > 0:
            days_in += 1
        prev_pos = pos

    total = (equity - 1.0) * 100.0
    mean_r = sum_ret / n
    var_r = sum_ret_sq / n - mean_r * mean_r
    std_r = var_r ** 0.5 if var_r > 0.0 else 1e-10
    sharpe = mean_r / std_r * (252.0 ** 0.5)
    pct_in = days_in / n * 100.0
    return sharpe, total, max_dd * 100.0, pct_in, float(n_trans)


@nb.njit(cache=True)
def _bt_enhanced_full(ivol, ret, baseline, ema_f, ema_s,
                      vol_spread, vol_z, mom_signal, intraday_range,
                      spike_pct, exit_confirm, cooldown, confirm,
                      spread_exit_thr, vol_z_exit_thr, mom_entry_thr,
                      range_exit_mult):
    """Full arrays version for charting."""
    spike_mult = 1.0 + spike_pct / 100.0
    n = len(ivol)
    positions = np.empty(n, dtype=np.float64)
    exit_sig  = np.empty(n, dtype=np.float64)
    entry_sig = np.empty(n, dtype=np.float64)
    exit_level = np.empty(n, dtype=np.float64)

    pos = 1
    days_since_entry = cooldown
    days_ema_below = 0
    days_above_spike = 0

    range_buf = np.empty(20)
    range_buf[:] = 0.0
    range_idx = 0
    range_sum = 0.0

    for i in range(n):
        iv = ivol[i]
        thr_i = baseline[i] * spike_mult

        old_val = range_buf[range_idx]
        range_buf[range_idx] = intraday_range[i]
        range_sum = range_sum - old_val + intraday_range[i]
        range_idx = (range_idx + 1) % 20
        range_mean = range_sum / min(i + 1, 20)

        if iv >= thr_i:
            days_above_spike += 1
        else:
            days_above_spike = 0

        if ema_f[i] < ema_s[i]:
            days_ema_below += 1
        else:
            days_ema_below = 0

        # Exit conditions
        exit_triggered = False
        if days_above_spike >= exit_confirm and days_since_entry >= cooldown:
            exit_triggered = True
        if spread_exit_thr > 0.0 and vol_spread[i] > spread_exit_thr:
            exit_triggered = True
        if vol_z_exit_thr > 0.0 and vol_z[i] > vol_z_exit_thr and iv > baseline[i]:
            exit_triggered = True
        if range_exit_mult > 0.0 and i >= 20:
            if intraday_range[i] > range_mean * range_exit_mult and iv > baseline[i]:
                exit_triggered = True

        exit_sig[i] = 0.0 if exit_triggered else 1.0

        # Entry conditions
        entry_ok = days_ema_below >= confirm
        if mom_entry_thr > -900.0 and mom_signal[i] < mom_entry_thr:
            entry_ok = False
        entry_sig[i] = 1.0 if entry_ok else 0.0
        exit_level[i] = thr_i

        old_pos = pos
        if pos == 1 and exit_triggered and days_since_entry >= cooldown:
            pos = 0
        elif pos == 0 and entry_ok:
            pos = 1
            days_since_entry = 0
        if pos == 1:
            days_since_entry += 1
        positions[i] = float(pos)

    return positions, exit_sig, entry_sig, exit_level


# ── Pre-compute arrays for enhanced backtest ──
enh_ivol  = enh_clean['ivol'].values.astype(np.float64)
enh_ret   = enh_clean['ret'].values.astype(np.float64)
enh_vspread10 = enh_clean['vol_spread_10'].values.astype(np.float64)
enh_vspread20 = enh_clean['vol_spread_20'].values.astype(np.float64)
enh_volz20 = enh_clean['vol_z_20'].fillna(0).values.astype(np.float64)
enh_volz50 = enh_clean['vol_z_50'].fillna(0).values.astype(np.float64)
enh_mom2050 = enh_clean['mom_20_50'].values.astype(np.float64)
enh_mom20100 = enh_clean['mom_20_100'].values.astype(np.float64)
enh_range = enh_clean['intraday_range'].values.astype(np.float64)

# EMA cache on enhanced dataset
enh_ema_cache = {}
for span in all_spans:
    enh_ema_cache[span] = enh_clean['ivol'].ewm(span=span).mean().values.astype(np.float64)

# Warm up
_ = _bt_enhanced(enh_ivol, enh_ret, enh_ema_cache[60], enh_ema_cache[5], enh_ema_cache[20],
                 enh_vspread20, enh_volz20, enh_mom2050, enh_range,
                 30.0, 3, 15, 5, 0.0, 0.0, -999.0, 0.0)
_ = _bt_enhanced_full(enh_ivol, enh_ret, enh_ema_cache[60], enh_ema_cache[5], enh_ema_cache[20],
                      enh_vspread20, enh_volz20, enh_mom2050, enh_range,
                      30.0, 3, 15, 5, 0.0, 0.0, -999.0, 0.0)
print("Enhanced numba kernels compiled ✓")

# Quick sanity: enhanced with all filters disabled should match basic
r_base = _bt_enhanced(enh_ivol, enh_ret, enh_ema_cache[60], enh_ema_cache[5], enh_ema_cache[20],
                      enh_vspread20, enh_volz20, enh_mom2050, enh_range,
                      30.0, 3, 15, 5,
                      0.0, 0.0, -999.0, 0.0)  # all filters off
print(f"Baseline (filters off): Sharpe={r_base[0]:.3f}  Total={r_base[1]:+.1f}%  MaxDD={r_base[2]:.1f}%  In={r_base[3]:.1f}%")

Enhanced numba kernels compiled ✓
Baseline (filters off): Sharpe=0.225  Total=+59.8%  MaxDD=-52.8%  In=96.4%


In [17]:
# ── Phase 1: Coarse grid scan over full multi-signal space ──
import time

# IVol core params (coarser than before to budget for new dimensions)
p2_exit_ema     = [20, 40, 60, 80, 100]
p2_spike_pct    = [15, 25, 35, 50, 70]
p2_exit_confirm = [1, 3, 5, 7]
p2_cooldown     = [5, 15, 30]
p2_ema_fast     = [3, 5, 8]
p2_ema_slow     = [15, 25, 35]
p2_confirm      = [2, 5, 10]

# NEW signal filters
p2_spread_thr   = [0, 5, 10, 15, 20]        # 0 = disabled
p2_volz_thr     = [0, 1.5, 2.5, 3.5]        # 0 = disabled
p2_mom_thr      = [-999, -0.03, 0, 0.02]    # -999 = disabled
p2_range_mult   = [0, 2.5, 3.5]             # 0 = disabled

# Choose vol_spread and vol_z variants
spread_variants = [(enh_vspread10, 'sp10'), (enh_vspread20, 'sp20')]
volz_variants   = [(enh_volz20, 'vz20'), (enh_volz50, 'vz50')]
mom_variants    = [(enh_mom2050, 'mom2050'), (enh_mom20100, 'mom20100')]

from itertools import product

# Core IVol combos
core_combos = list(product(p2_exit_ema, p2_spike_pct, p2_exit_confirm,
                           p2_cooldown, p2_ema_fast, p2_ema_slow, p2_confirm))

# Filter combos
filter_combos = list(product(p2_spread_thr, p2_volz_thr, p2_mom_thr, p2_range_mult))

total = len(core_combos) * len(filter_combos)
print(f"Full grid: {len(core_combos):,} core × {len(filter_combos):,} filters = {total:,}")

# Use spread_20, vol_z_20, mom_20_50 as primary variant for Phase 1
t0 = time.perf_counter()
n_cols = 16
results_enh = np.empty((total, n_cols), dtype=np.float64)
idx = 0

for ex_ema, sp, ex_cf, cd, ef, es, cf in core_combos:
    base_arr = enh_ema_cache[ex_ema]
    ema_f_arr = enh_ema_cache[ef]
    ema_s_arr = enh_ema_cache[es]
    for spr_thr, vz_thr, mom_thr, rng_mult in filter_combos:
        sh, tot, mdd, pin, ntr = _bt_enhanced(
            enh_ivol, enh_ret, base_arr, ema_f_arr, ema_s_arr,
            enh_vspread20, enh_volz20, enh_mom2050, enh_range,
            float(sp), ex_cf, cd, cf,
            float(spr_thr), float(vz_thr), float(mom_thr), float(rng_mult))
        results_enh[idx] = [ex_ema, sp, ex_cf, cd, ef, es, cf,
                            spr_thr, vz_thr, mom_thr, rng_mult,
                            sh, tot, mdd, pin, ntr]
        idx += 1

elapsed = time.perf_counter() - t0
print(f"Phase 1 completed: {idx:,} backtests in {elapsed:.1f}s ({elapsed/idx*1e6:.1f} μs/bt)")

opt_enh = pd.DataFrame(results_enh[:idx], columns=[
    'exit_ema', 'spike_pct', 'exit_confirm', 'cooldown', 'ema_fast', 'ema_slow', 'confirm',
    'spread_thr', 'volz_thr', 'mom_thr', 'range_mult',
    'sharpe', 'total', 'maxdd', 'pct_in', 'n_trans'
])
for c in ['exit_ema','spike_pct','exit_confirm','cooldown','ema_fast','ema_slow','confirm','n_trans']:
    opt_enh[c] = opt_enh[c].astype(int)

# ── Filter viable & rank ──
viable_enh = opt_enh[(opt_enh['pct_in'] > 40) & (opt_enh['sharpe'] > 0)].copy()
viable_enh['score'] = viable_enh['sharpe'] - (viable_enh['maxdd'].abs() / 100) * 0.5

show_cols_enh = ['exit_ema','spike_pct','exit_confirm','cooldown',
                 'ema_fast','ema_slow','confirm',
                 'spread_thr','volz_thr','mom_thr','range_mult',
                 'sharpe','total','maxdd','pct_in','n_trans']
fmt_enh = {'sharpe':'{:.3f}','total':'{:+.1f}%','maxdd':'{:.1f}%','pct_in':'{:.0f}%',
           'mom_thr':'{:.3f}','range_mult':'{:.1f}','volz_thr':'{:.1f}','spread_thr':'{:.0f}'}

# Dark-theme-friendly bar colors
_clr_sharpe = '#66ffcc'   # mint green
_clr_total  = '#8cb4ff'   # soft blue
_clr_score  = '#ffb347'   # warm amber

print(f"\n{'='*110}")
print(f"Viable (in-market > 40%, Sharpe > 0): {len(viable_enh):,} / {len(opt_enh):,}")
print(f"{'='*110}")

# How many of the top results use the NEW filters vs. none?
top50 = viable_enh.nlargest(50, 'sharpe')
uses_new_top50 = ((top50['spread_thr'] > 0) | (top50['volz_thr'] > 0) |
                  (top50['mom_thr'] > -900) | (top50['range_mult'] > 0))
print(f"\nOf top-50 by Sharpe: {uses_new_top50.sum()} use new signal filters, {(~uses_new_top50).sum()} are IVol-only")

print(f"\nTop 25 by Sharpe (enhanced):")
display(viable_enh.nlargest(25, 'sharpe')[show_cols_enh].reset_index(drop=True)
    .style.format(fmt_enh)
    .bar(subset=['sharpe'], color=_clr_sharpe)
    .bar(subset=['total'], color=_clr_total)
    .background_gradient(subset=['maxdd'], cmap='RdYlGn'))

print(f"\nTop 15 by Total Return (enhanced):")
display(viable_enh.nlargest(15, 'total')[show_cols_enh].reset_index(drop=True)
    .style.format(fmt_enh).bar(subset=['total'], color=_clr_total))

print(f"\nTop 15 Balanced — Sharpe − 0.5×|MaxDD| (enhanced):")
display(viable_enh.nlargest(15, 'score')[show_cols_enh + ['score']].reset_index(drop=True)
    .style.format({**fmt_enh, 'score':'{:.3f}'}).bar(subset=['score'], color=_clr_score))

# ── Show improvement from new filters ──
best_basic = opt_enh[(opt_enh['spread_thr']==0) & (opt_enh['volz_thr']==0) &
                      (opt_enh['mom_thr']==-999) & (opt_enh['range_mult']==0) &
                      (opt_enh['pct_in']>40) & (opt_enh['sharpe']>0)]
# Filter viable_enh for rows that use at least one new signal
uses_new_all = ((viable_enh['spread_thr'] > 0) | (viable_enh['volz_thr'] > 0) |
                (viable_enh['mom_thr'] > -900) | (viable_enh['range_mult'] > 0))
best_enhanced = viable_enh[uses_new_all]

print(f"\n{'─'*80}")
if len(best_basic) > 0 and len(best_enhanced) > 0:
    bb = best_basic.nlargest(1, 'sharpe').iloc[0]
    be = viable_enh.nlargest(1, 'sharpe').iloc[0]
    print(f"Best IVol-only:  Sharpe={bb['sharpe']:.3f}  Total={bb['total']:+.1f}%  MaxDD={bb['maxdd']:.1f}%")
    print(f"Best enhanced:   Sharpe={be['sharpe']:.3f}  Total={be['total']:+.1f}%  MaxDD={be['maxdd']:.1f}%")
    print(f"Δ Sharpe: {be['sharpe'] - bb['sharpe']:+.3f}")

Full grid: 8,100 core × 240 filters = 1,944,000
Phase 1 completed: 1,944,000 backtests in 37.1s (19.1 μs/bt)

Viable (in-market > 40%, Sharpe > 0): 721,217 / 1,944,000

Of top-50 by Sharpe: 50 use new signal filters, 0 are IVol-only

Top 25 by Sharpe (enhanced):


,exit_ema,spike_pct,exit_confirm,cooldown,ema_fast,ema_slow,confirm,spread_thr,volz_thr,mom_thr,range_mult,sharpe,total,maxdd,pct_in,n_trans
0,60,15,1,15,8,35,10,5,3.5,-0.030,2.5,0.303,+76.5%,-25.4%,56%,280
1,80,15,1,15,8,35,10,5,3.5,-0.030,2.5,0.303,+76.5%,-25.4%,56%,280
2,40,15,1,15,8,35,10,5,3.5,-0.030,2.5,0.302,+76.2%,-25.4%,56%,280
3,20,15,1,15,8,35,10,5,3.5,-0.030,2.5,0.301,+76.0%,-25.4%,56%,282
4,20,15,1,15,8,35,10,5,3.5,-0.030,0.0,0.300,+75.5%,-25.4%,56%,278
5,20,15,1,15,8,35,10,5,3.5,-0.030,3.5,0.300,+75.5%,-25.4%,56%,278
6,60,15,1,15,8,35,10,5,3.5,-0.030,0.0,0.300,+75.5%,-25.4%,56%,278
7,60,15,1,15,8,35,10,5,3.5,-0.030,3.5,0.300,+75.5%,-25.4%,56%,278
8,80,15,1,15,8,35,10,5,3.5,-0.030,0.0,0.300,+75.5%,-25.4%,56%,278
9,80,15,1,15,8,35,10,5,3.5,-0.030,3.5,0.300,+75.5%,-25.4%,56%,278



Top 15 by Total Return (enhanced):


,exit_ema,spike_pct,exit_confirm,cooldown,ema_fast,ema_slow,confirm,spread_thr,volz_thr,mom_thr,range_mult,sharpe,total,maxdd,pct_in,n_trans
0,100,35,7,5,3,35,10,0,0.0,0.000,0.0,0.286,+95.7%,-40.3%,93%,8
1,100,35,7,5,3,35,10,20,0.0,0.000,0.0,0.286,+95.7%,-40.3%,93%,8
2,100,35,7,5,5,15,10,0,0.0,0.000,0.0,0.286,+95.7%,-40.3%,93%,8
3,100,35,7,5,5,15,10,20,0.0,0.000,0.0,0.286,+95.7%,-40.3%,93%,8
4,100,35,7,5,5,25,10,0,0.0,0.000,0.0,0.286,+95.7%,-40.3%,93%,8
5,100,35,7,5,5,25,10,20,0.0,0.000,0.0,0.286,+95.7%,-40.3%,93%,8
6,100,35,7,5,8,15,10,0,0.0,0.000,0.0,0.286,+95.7%,-40.3%,93%,8
7,100,35,7,5,8,15,10,20,0.0,0.000,0.0,0.286,+95.7%,-40.3%,93%,8
8,100,35,7,15,3,35,10,0,0.0,0.000,0.0,0.286,+95.7%,-40.3%,93%,8
9,100,35,7,15,3,35,10,20,0.0,0.000,0.0,0.286,+95.7%,-40.3%,93%,8



Top 15 Balanced — Sharpe − 0.5×|MaxDD| (enhanced):


,exit_ema,spike_pct,exit_confirm,cooldown,ema_fast,ema_slow,confirm,spread_thr,volz_thr,mom_thr,range_mult,sharpe,total,maxdd,pct_in,n_trans,score
0,60,15,1,15,8,35,10,5,3.5,-0.030,2.5,0.303,+76.5%,-25.4%,56%,280,0.176
1,80,15,1,15,8,35,10,5,3.5,-0.030,2.5,0.303,+76.5%,-25.4%,56%,280,0.176
2,40,15,1,15,8,35,10,5,3.5,-0.030,2.5,0.302,+76.2%,-25.4%,56%,280,0.175
3,20,15,1,15,8,35,10,5,3.5,-0.030,2.5,0.301,+76.0%,-25.4%,56%,282,0.174
4,20,15,1,15,8,35,10,5,3.5,-0.030,0.0,0.300,+75.5%,-25.4%,56%,278,0.173
5,20,15,1,15,8,35,10,5,3.5,-0.030,3.5,0.300,+75.5%,-25.4%,56%,278,0.173
6,60,15,1,15,8,35,10,5,3.5,-0.030,0.0,0.300,+75.5%,-25.4%,56%,278,0.173
7,60,15,1,15,8,35,10,5,3.5,-0.030,3.5,0.300,+75.5%,-25.4%,56%,278,0.173
8,80,15,1,15,8,35,10,5,3.5,-0.030,0.0,0.300,+75.5%,-25.4%,56%,278,0.173
9,80,15,1,15,8,35,10,5,3.5,-0.030,3.5,0.300,+75.5%,-25.4%,56%,278,0.173



────────────────────────────────────────────────────────────────────────────────
Best IVol-only:  Sharpe=0.264  Total=+83.6%  MaxDD=-50.4%
Best enhanced:   Sharpe=0.303  Total=+76.5%  MaxDD=-25.4%
Δ Sharpe: +0.038


In [19]:
# ── Phase 2: Fine-tune around top-50 seeds (deduped flat grid) ───────────────
import time
from itertools import product

N_SEEDS = 50
seeds = viable_enh.nlargest(N_SEEDS, 'sharpe')[
    ['exit_ema','spike_pct','exit_confirm','cooldown','ema_fast','ema_slow','confirm',
     'spread_thr','volz_thr','mom_thr','range_mult']].values

def _perturb_int(v, deltas, lo, hi):
    return sorted({max(lo, min(hi, int(v + d))) for d in deltas})

def _perturb_flt(v, deltas, lo=None):
    vals = {round(v + d, 4) for d in deltas}
    return sorted(vals if lo is None else {max(lo, x) for x in vals})

# Collect ALL unique combo tuples across seeds (dedup BEFORE running)
combo_set = set()
for s in seeds:
    ex0, sp0, xc0, cd0, ef0, es0, cf0 = [int(x) for x in s[:7]]
    spr0, vz0, mom0, rng0 = s[7], s[8], s[9], s[10]

    for combo in product(
        _perturb_int(ex0, [-10, 0, 10], 10, 120),
        _perturb_int(sp0,  [-5,  0,  5],  5, 100),
        _perturb_int(xc0,  [-1,  0,  1],  1,  10),
        _perturb_int(cd0,  [-5,  0,  5],  1,  60),
        _perturb_int(ef0,  [-1,  0,  1],  2,  15),
        _perturb_int(es0,  [-5,  0,  5], 10,  40),
        _perturb_int(cf0,  [-1,  0,  1],  1,  20),
        _perturb_flt(spr0, [-3, 0, 3], lo=0),
        _perturb_flt(vz0,  [-0.5, 0, 0.5], lo=0),
        _perturb_flt(mom0, [-0.01, 0, 0.01]),
        _perturb_flt(rng0, [-0.5, 0, 0.5], lo=0),
    ):
        ex, sp, xc, cd, ef, es, cf = combo[:7]
        if ef >= es:
            continue
        if enh_ema_cache.get(ex) is None or enh_ema_cache.get(ef) is None or enh_ema_cache.get(es) is None:
            continue
        combo_set.add(combo)

combos = sorted(combo_set)
print(f"Phase 2: {len(combos):,} unique combos (from {N_SEEDS} seeds, deduped)")

# ── Flat loop (same pattern as Phase 1) ──
t0 = time.perf_counter()
n_cols = 16
results_p2 = np.empty((len(combos), n_cols), dtype=np.float64)
idx = 0

for ex, sp, xc, cd, ef, es, cf, spr, vz, mom, rng in combos:
    sh, tot, mdd, pin, ntr = _bt_enhanced(
        enh_ivol, enh_ret, enh_ema_cache[ex], enh_ema_cache[ef], enh_ema_cache[es],
        enh_vspread20, enh_volz20, enh_mom2050, enh_range,
        float(sp), int(xc), int(cd), int(cf),
        float(spr), float(vz), float(mom), float(rng))
    results_p2[idx] = [ex, sp, xc, cd, ef, es, cf, spr, vz, mom, rng,
                       sh, tot, mdd, pin, ntr]
    idx += 1

elapsed = time.perf_counter() - t0
print(f"Phase 2: {idx:,} backtests in {elapsed:.1f}s ({elapsed/idx*1e6:.1f} μs/bt)")

# Build DataFrame & filter
fine_df = pd.DataFrame(results_p2[:idx], columns=[
    'exit_ema','spike_pct','exit_confirm','cooldown','ema_fast','ema_slow','confirm',
    'spread_thr','volz_thr','mom_thr','range_mult',
    'sharpe','total','maxdd','pct_in','n_trans'])
for c in ['exit_ema','spike_pct','exit_confirm','cooldown','ema_fast','ema_slow','confirm','n_trans']:
    fine_df[c] = fine_df[c].astype(int)

fine_dedup = fine_df[(fine_df['pct_in'] > 40) & (fine_df['sharpe'] > 0)].copy()
fine_dedup['score'] = fine_dedup['sharpe'] - (fine_dedup['maxdd'].abs() / 100) * 0.5
fine_dedup = fine_dedup.sort_values('sharpe', ascending=False)

print(f"Phase 2 viable: {len(fine_dedup):,} / {len(fine_df):,}")

print(f"\n{'='*110}")
print("FINAL Top 25 by Sharpe (Phase 2 fine-tune):")
display(fine_dedup.nlargest(25, 'sharpe')[show_cols_enh].reset_index(drop=True)
    .style.format(fmt_enh)
    .bar(subset=['sharpe'], color=_clr_sharpe)
    .bar(subset=['total'], color=_clr_total)
    .background_gradient(subset=['maxdd'], cmap='RdYlGn'))

print(f"\nFINAL Top 15 Balanced — Sharpe − 0.5×|MaxDD|:")
display(fine_dedup.nlargest(15, 'score')[show_cols_enh + ['score']].reset_index(drop=True)
    .style.format({**fmt_enh, 'score':'{:.3f}'}).bar(subset=['score'], color=_clr_score))

# ── Compare Phase 1 vs Phase 2 ──
p1_best = viable_enh.nlargest(1, 'sharpe').iloc[0]
p2_best = fine_dedup.nlargest(1, 'sharpe').iloc[0]
print(f"\n{'─'*80}")
print(f"Phase 1 best: Sharpe={p1_best['sharpe']:.3f}  Total={p1_best['total']:+.1f}%  MaxDD={p1_best['maxdd']:.1f}%")
print(f"Phase 2 best: Sharpe={p2_best['sharpe']:.3f}  Total={p2_best['total']:+.1f}%  MaxDD={p2_best['maxdd']:.1f}%")
print(f"Improvement:  ΔSharpe={p2_best['sharpe'] - p1_best['sharpe']:+.3f}  ΔTotal={p2_best['total'] - p1_best['total']:+.1f}%")

Phase 2: 3,621,672 unique combos (from 50 seeds, deduped)
Phase 2: 3,621,672 backtests in 69.2s (19.1 μs/bt)
Phase 2 viable: 1,510,946 / 3,621,672

FINAL Top 25 by Sharpe (Phase 2 fine-tune):


,exit_ema,spike_pct,exit_confirm,cooldown,ema_fast,ema_slow,confirm,spread_thr,volz_thr,mom_thr,range_mult,sharpe,total,maxdd,pct_in,n_trans
0,110,10,8,15,8,30,9,5,3.5,-0.020,0.0,0.399,+122.8%,-21.3%,57%,278
1,60,15,8,15,8,30,9,5,3.5,-0.020,4.0,0.399,+122.8%,-21.3%,57%,278
2,60,15,8,15,8,30,9,5,3.5,-0.020,3.0,0.399,+122.8%,-21.3%,57%,278
3,30,15,8,15,8,30,9,5,3.5,-0.020,3.5,0.399,+122.8%,-21.3%,57%,278
4,80,15,8,15,8,30,9,5,3.5,-0.020,3.5,0.399,+122.8%,-21.3%,57%,278
5,80,15,8,15,8,30,9,5,3.5,-0.020,0.0,0.399,+122.8%,-21.3%,57%,278
6,30,15,8,15,8,30,9,5,3.5,-0.020,0.0,0.399,+122.8%,-21.3%,57%,278
7,30,15,8,15,8,30,9,5,3.5,-0.020,4.0,0.399,+122.8%,-21.3%,57%,278
8,80,15,8,15,8,30,9,5,3.5,-0.020,3.0,0.399,+122.8%,-21.3%,57%,278
9,110,15,7,15,8,30,9,5,3.5,-0.020,3.5,0.399,+122.8%,-21.3%,57%,278



FINAL Top 15 Balanced — Sharpe − 0.5×|MaxDD|:


,exit_ema,spike_pct,exit_confirm,cooldown,ema_fast,ema_slow,confirm,spread_thr,volz_thr,mom_thr,range_mult,sharpe,total,maxdd,pct_in,n_trans,score
0,110,10,8,15,8,30,9,5,3.5,-0.020,0.0,0.399,+122.8%,-21.3%,57%,278,0.293
1,60,15,8,15,8,30,9,5,3.5,-0.020,4.0,0.399,+122.8%,-21.3%,57%,278,0.293
2,60,15,8,15,8,30,9,5,3.5,-0.020,3.0,0.399,+122.8%,-21.3%,57%,278,0.293
3,30,15,8,15,8,30,9,5,3.5,-0.020,3.5,0.399,+122.8%,-21.3%,57%,278,0.293
4,80,15,8,15,8,30,9,5,3.5,-0.020,3.5,0.399,+122.8%,-21.3%,57%,278,0.293
5,80,15,8,15,8,30,9,5,3.5,-0.020,0.0,0.399,+122.8%,-21.3%,57%,278,0.293
6,30,15,8,15,8,30,9,5,3.5,-0.020,0.0,0.399,+122.8%,-21.3%,57%,278,0.293
7,30,15,8,15,8,30,9,5,3.5,-0.020,4.0,0.399,+122.8%,-21.3%,57%,278,0.293
8,80,15,8,15,8,30,9,5,3.5,-0.020,3.0,0.399,+122.8%,-21.3%,57%,278,0.293
9,110,15,7,15,8,30,9,5,3.5,-0.020,3.5,0.399,+122.8%,-21.3%,57%,278,0.293



────────────────────────────────────────────────────────────────────────────────
Phase 1 best: Sharpe=0.303  Total=+76.5%  MaxDD=-25.4%
Phase 2 best: Sharpe=0.399  Total=+122.8%  MaxDD=-21.3%
Improvement:  ΔSharpe=+0.096  ΔTotal=+46.3%


In [20]:
# ── Enhanced interactive chart: 11 sliders, 5-panel chart  ───────────────────
# ── Dark navy theme (same palette as cell 8) ─────────────────────────────────
_BG2     = '#111827'   # dark navy (paper)
_PBG2    = '#1a2235'   # slightly lighter (plot area)
_GRID2   = '#1e293b'   # subtle grid lines

_E_LONG  = '#60a5fa'   # blue-400
_E_FLAT  = '#f87171'   # red-400
_E_STRAT = '#34d399'   # emerald
_E_BH    = '#6b7280'   # gray-500
_E_IVOL  = '#fb923c'   # orange-400
_E_RVOL  = '#fbbf24'   # amber-400
_E_EXIT  = '#38bdf8'   # sky-400
_E_SPRD  = '#a78bfa'   # violet-400 – spread
_E_VOLZ  = '#c084fc'   # purple-400 – vol z-score
_E_RNG   = '#5eead4'   # teal-300   – intraday range
_E_MOM   = '#93c5fd'   # blue-300   – momentum
_E_ESIG  = '#f87171'   # red fill
_E_NSIG  = '#34d399'   # green fill

# Auto-load best params from Phase 2 (or Phase 1 fallback)
if 'fine_dedup' in dir() and len(fine_dedup) > 0:
    _eb = fine_dedup.nlargest(1, 'sharpe').iloc[0]
elif 'viable_enh' in dir() and len(viable_enh) > 0:
    _eb = viable_enh.nlargest(1, 'sharpe').iloc[0]
else:
    _eb = None

if _eb is not None:
    _ebp = {k: int(_eb[k]) for k in ['exit_ema','spike_pct','exit_confirm','cooldown',
            'ema_fast','ema_slow','confirm']}
    _ebp['spread_thr'] = float(_eb['spread_thr'])
    _ebp['volz_thr']   = float(_eb['volz_thr'])
    _ebp['mom_thr']    = float(_eb['mom_thr'])
    _ebp['range_mult'] = float(_eb['range_mult'])
else:
    _ebp = dict(exit_ema=60, spike_pct=30, exit_confirm=3, cooldown=15,
                ema_fast=5, ema_slow=20, confirm=5,
                spread_thr=0, volz_thr=0, mom_thr=-999, range_mult=0)
print(f"Enhanced slider defaults → best result: {_ebp}")

def _build_enh_fig(exit_ema, spike_pct, exit_cf, cooldown, ef, es, cf,
                   spread_thr, volz_thr, mom_thr, range_mult):
    """Build 5-panel chart with all signals visible."""
    baseline = enh_clean['ivol'].ewm(span=exit_ema).mean().values.astype(np.float64)
    ema_f_arr = enh_clean['ivol'].ewm(span=ef).mean().values.astype(np.float64)
    ema_s_arr = enh_clean['ivol'].ewm(span=es).mean().values.astype(np.float64)

    positions, exit_sig, entry_sig, exit_lev = _bt_enhanced_full(
        enh_ivol, enh_ret, baseline, ema_f_arr, ema_s_arr,
        enh_vspread20, enh_volz20, enh_mom2050, enh_range,
        float(spike_pct), exit_cf, cooldown, cf,
        float(spread_thr), float(volz_thr), float(mom_thr), float(range_mult))

    res = enh_clean[['close', 'ret', 'ivol']].copy()
    res['position'] = positions
    res['exit_sig'] = exit_sig
    res['entry_sig'] = entry_sig
    res['exit_level'] = exit_lev
    res['strat_ret'] = pd.Series(positions, index=res.index).shift(1).fillna(1) * res['ret']
    res['equity_strat'] = (1 + res['strat_ret']).cumprod()

    exp = pd.Series(positions, index=res.index).shift(1).fillna(1)
    in_mask = exp > 0
    sr = res['strat_ret']
    eq = res['equity_strat']
    sharpe = sr.mean() / sr.std() * np.sqrt(252) if sr.std() > 0 else 0
    mdd = ((eq - eq.cummax()) / eq.cummax()).min()
    total = (eq.iloc[-1] - 1) * 100
    pct_in = (exp > 0).sum() / len(res) * 100
    n_tr = (pd.Series(positions).diff().fillna(0) != 0).sum()
    stats = dict(sharpe=round(sharpe, 3), total=round(total, 1),
                 maxdd=round(mdd * 100, 1), pct_in=round(pct_in, 1), n_tr=int(n_tr))

    fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.03,
        row_heights=[0.25, 0.15, 0.22, 0.20, 0.18])

    # Row 1: Price
    fig.add_trace(go.Scatter(x=res.index, y=res['close'],
        line=dict(color='#374151', width=1), showlegend=False, hoverinfo='skip'), row=1, col=1)
    fig.add_trace(go.Scatter(x=res.index, y=res['close'].where(in_mask),
        name='Long', line=dict(color=_E_LONG, width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=res.index, y=res['close'].where(~in_mask),
        name='Flat', line=dict(color=_E_FLAT, width=2)), row=1, col=1)

    # Row 2: Equity curves
    fig.add_trace(go.Scatter(x=res.index, y=enh_bh_eq.loc[res.index], name='Buy & Hold',
        line=dict(color=_E_BH, width=1.5, dash='dash')), row=2, col=1)
    fig.add_trace(go.Scatter(x=res.index, y=eq, name='Strategy',
        line=dict(color=_E_STRAT, width=2)), row=2, col=1)

    # Row 3: IVol + RVol + Exit level
    fig.add_trace(go.Scatter(x=res.index, y=enh_clean['ivol'], name='IVol',
        line=dict(color=_E_IVOL, width=1, dash='dot')), row=3, col=1)
    fig.add_trace(go.Scatter(x=res.index, y=enh_clean['rvol_park_20'],
        name='RVol (Park 20d)', line=dict(color=_E_RVOL, width=1.2)), row=3, col=1)
    fig.add_trace(go.Scatter(x=res.index, y=exit_lev,
        name='Exit level', line=dict(color=_E_EXIT, width=1.5, dash='dash')), row=3, col=1)
    if spread_thr > 0:
        fig.add_trace(go.Scatter(x=res.index, y=enh_clean['vol_spread_20'],
            name='IVol-RVol spread', line=dict(color=_E_SPRD, width=1.2)), row=3, col=1)

    # Row 4: Volume z-score + intraday range
    fig.add_trace(go.Scatter(x=res.index, y=enh_clean['vol_z_20'],
        name='Vol z-score (20d)', line=dict(color=_E_VOLZ, width=1.2)), row=4, col=1)
    fig.add_trace(go.Scatter(x=res.index, y=enh_clean['intraday_range'],
        name='Intraday range %', line=dict(color=_E_RNG, width=1)), row=4, col=1)
    if volz_thr > 0:
        fig.add_hline(y=volz_thr, line_dash="dash", line_color=_E_FLAT,
                      annotation_text=f"vol_z exit={volz_thr}", row=4, col=1)

    # Row 5: Momentum + signals
    fig.add_trace(go.Scatter(x=res.index, y=enh_clean['mom_20_50'] * 100,
        name='Mom 20/50 (%)', line=dict(color=_E_MOM, width=1.2)), row=5, col=1)
    if mom_thr > -900:
        fig.add_hline(y=mom_thr * 100, line_dash="dash", line_color=_E_NSIG,
                      annotation_text=f"entry mom>={mom_thr:.0%}", row=5, col=1)
    fig.add_trace(go.Scatter(x=res.index, y=exit_sig * 2 - 3.5,
        name='Exit signal', line=dict(color=_E_ESIG, width=1),
        fill='tozeroy', fillcolor='rgba(248,113,113,0.08)'), row=5, col=1)

    # Active filters text
    filters_on = []
    if spread_thr > 0: filters_on.append(f'SpreadExit>{spread_thr:.0f}')
    if volz_thr > 0: filters_on.append(f'VolZ>{volz_thr:.1f}')
    if mom_thr > -900: filters_on.append(f'MomEntry>{mom_thr:.1%}')
    if range_mult > 0: filters_on.append(f'RangeMult>{range_mult:.1f}')
    filters_text = ' + '.join(filters_on) if filters_on else 'none'

    title_text = (
        f"<b>Exit: EMA-{exit_ema}×{1+spike_pct/100:.0%} {exit_cf}d (cd={cooldown})  |  "
        f"Entry: EMA-{ef}<{es} {cf}d  |  Filters: {filters_text}</b><br>"
        f"<span style='font-size:13px; color:#94a3b8'>"
        f"Sharpe <b style='color:#34d399'>{stats['sharpe']:.3f}</b>  ·  "
        f"Total <b style='color:#60a5fa'>{stats['total']:+.1f}%</b>  ·  "
        f"MaxDD <b style='color:#f87171'>{stats['maxdd']:.1f}%</b>  ·  "
        f"In {stats['pct_in']:.0f}%  ·  "
        f"Trades {stats['n_tr']}"
        f"&nbsp;&nbsp;│&nbsp;&nbsp;"
        f"B&H: {enh_bh_sharpe:.3f} / {enh_bh_total:+.1f}% / {enh_bh_mdd:.1f}%</span>"
    )

    fig.update_layout(
        height=1200, hovermode='x unified',
        title=dict(text=title_text, font=dict(size=14, color='#e2e8f0')),
        legend=dict(orientation='h', y=-0.04, font=dict(color='#94a3b8')),
        margin=dict(t=100, b=60),
        paper_bgcolor=_BG2, plot_bgcolor=_PBG2,
        font=dict(color='#94a3b8'))
    for r in range(1, 6):
        fig.update_xaxes(gridcolor=_GRID2, zeroline=False, row=r, col=1)
        fig.update_yaxes(gridcolor=_GRID2, zeroline=False, row=r, col=1)
    fig.update_yaxes(title_text='Price', row=1, col=1)
    fig.update_yaxes(title_text='Equity', type='log', row=2, col=1)
    fig.update_yaxes(title_text='Vol Levels', row=3, col=1)
    fig.update_yaxes(title_text='Vol z / Range', row=4, col=1)
    fig.update_yaxes(title_text='Momentum / Sig', row=5, col=1)
    return fig

# ── Sliders (defaults = best enhanced result) ──
w2 = {'description_width': '160px'}
ly2 = widgets.Layout(width='92%')

sl2_exit_ema = widgets.IntSlider(value=_ebp['exit_ema'], min=10, max=120, step=5,
    description='Exit baseline EMA', continuous_update=False, style=w2, layout=ly2)
sl2_spike = widgets.IntSlider(value=_ebp['spike_pct'], min=5, max=100, step=5,
    description='Spike % above EMA', continuous_update=False, style=w2, layout=ly2)
sl2_exit_cf = widgets.IntSlider(value=_ebp['exit_confirm'], min=1, max=10,
    description='Exit confirm days', continuous_update=False, style=w2, layout=ly2)
sl2_cooldown = widgets.IntSlider(value=_ebp['cooldown'], min=1, max=60,
    description='Cooldown days', continuous_update=False, style=w2, layout=ly2)
sl2_ema_fast = widgets.IntSlider(value=_ebp['ema_fast'], min=2, max=15,
    description='Re-entry EMA-fast', continuous_update=False, style=w2, layout=ly2)
sl2_ema_slow = widgets.IntSlider(value=_ebp['ema_slow'], min=10, max=40, step=5,
    description='Re-entry EMA-slow', continuous_update=False, style=w2, layout=ly2)
sl2_confirm = widgets.IntSlider(value=_ebp['confirm'], min=1, max=20,
    description='Re-entry confirm', continuous_update=False, style=w2, layout=ly2)

# NEW filter sliders
sl2_spread = widgets.FloatSlider(value=_ebp['spread_thr'], min=0, max=30, step=1,
    description='Vol spread exit thr', continuous_update=False, style=w2, layout=ly2)
sl2_volz = widgets.FloatSlider(value=_ebp['volz_thr'], min=0, max=5, step=0.5,
    description='Volume z-score exit', continuous_update=False, style=w2, layout=ly2)
sl2_mom = widgets.FloatSlider(value=_ebp['mom_thr'], min=-999, max=0.10, step=0.01,
    description='Mom entry thr (-999=off)', continuous_update=False, style=w2, layout=ly2,
    readout_format='.2f')
sl2_range = widgets.FloatSlider(value=_ebp['range_mult'], min=0, max=5, step=0.5,
    description='Range exit mult (0=off)', continuous_update=False, style=w2, layout=ly2)

out2 = widgets.Output()

def _update2(_=None):
    with out2:
        from IPython.display import clear_output
        clear_output(wait=True)
        _build_enh_fig(
            sl2_exit_ema.value, sl2_spike.value, sl2_exit_cf.value, sl2_cooldown.value,
            sl2_ema_fast.value, sl2_ema_slow.value, sl2_confirm.value,
            sl2_spread.value, sl2_volz.value, sl2_mom.value, sl2_range.value
        ).show()

for s in [sl2_exit_ema, sl2_spike, sl2_exit_cf, sl2_cooldown,
          sl2_ema_fast, sl2_ema_slow, sl2_confirm,
          sl2_spread, sl2_volz, sl2_mom, sl2_range]:
    s.observe(_update2, 'value')

# ── "Load Best" buttons for enhanced results ──
def _load_enh_best(rank=0):
    def _cb(_):
        src = fine_dedup if ('fine_dedup' in dir() and len(fine_dedup) > 0) else viable_enh
        row = src.nlargest(rank + 1, 'sharpe').iloc[rank]
        sl2_exit_ema.value = int(row['exit_ema'])
        sl2_spike.value = int(row['spike_pct'])
        sl2_exit_cf.value = int(row['exit_confirm'])
        sl2_cooldown.value = int(row['cooldown'])
        sl2_ema_fast.value = int(row['ema_fast'])
        sl2_ema_slow.value = int(row['ema_slow'])
        sl2_confirm.value = int(row['confirm'])
        sl2_spread.value = float(row['spread_thr'])
        sl2_volz.value = float(row['volz_thr'])
        sl2_mom.value = float(row['mom_thr'])
        sl2_range.value = float(row['range_mult'])
    return _cb

btn_eb1 = widgets.Button(description='▶ Best Sharpe', button_style='success',
    layout=widgets.Layout(width='140px'))
btn_eb2 = widgets.Button(description='▶ 2nd Best', button_style='info',
    layout=widgets.Layout(width='120px'))
btn_eb1.on_click(_load_enh_best(0))
btn_eb2.on_click(_load_enh_best(1))

_update2()

exit_box2 = widgets.VBox([
    widgets.HTML('<b style="font-size:13px; color:#f87171">EXIT parameters</b>'),
    sl2_exit_ema, sl2_spike, sl2_exit_cf, sl2_cooldown])
entry_box2 = widgets.VBox([
    widgets.HTML('<b style="font-size:13px; color:#34d399">RE-ENTRY parameters</b>'),
    sl2_ema_fast, sl2_ema_slow, sl2_confirm])
filter_box = widgets.VBox([
    widgets.HTML('<b style="font-size:13px; color:#60a5fa">SIGNAL FILTERS</b>'),
    sl2_spread, sl2_volz, sl2_mom, sl2_range])
btn_row2 = widgets.HBox([btn_eb1, btn_eb2],
    layout=widgets.Layout(gap='8px'))

display(widgets.VBox([btn_row2, exit_box2, entry_box2, filter_box, out2]))

Enhanced slider defaults → best result: {'exit_ema': 110, 'spike_pct': 10, 'exit_confirm': 8, 'cooldown': 15, 'ema_fast': 8, 'ema_slow': 30, 'confirm': 9, 'spread_thr': 5.0, 'volz_thr': 3.5, 'mom_thr': -0.02, 'range_mult': 0.0}
